In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "awasthiashu"
os.environ['KAGGLE_KEY'] = "e618870c938250f3a653d7e3965c06de"

!mkdir -p ~/.kaggle
!echo '{"username":"awasthiashu","key":"e618870c938250f3a653d7e3965c06de"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


In [2]:
# Update diffusers to source to support the latest training scripts
!pip install -q ultralytics transformers accelerate torch torchvision opencv-python pillow xformers
!pip install -q git+https://github.com/huggingface/diffusers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.6 MB/s eta 0:00:00:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 9.0 MB/s eta 0:00:00a 0:00:01


In [3]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image
import numpy as np
import cv2
from ultralytics import YOLO

# Setup Models
device = "cuda" if torch.cuda.is_available() else "cpu"
detection_model = YOLO('yolov8n.pt')

controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

def get_canny_image(image):
    image_np = np.array(image)
    image_canny = cv2.Canny(image_np, 100, 200)
    image_canny = image_canny[:, :, None]
    image_canny = np.concatenate([image_canny, image_canny, image_canny], axis=2)
    return Image.fromarray(image_canny)

def redesign_room(image_path, style="modern"):
    image = Image.open(image_path).convert("RGB")

    # Detection
    results = detection_model(image)
    labels = [detection_model.names[int(box.cls)] for box in results[0].boxes]
    objects_str = ", ".join(set(labels))

    # Prompting
    prompt = f"A professional photo of a {style} room with {objects_str}. 8k, realistic."

    # Generation
    canny = get_canny_image(image)
    result = pipe(prompt, image=canny, num_inference_steps=25).images[0]
    return result

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Example: Downloading ADE20K Subset (Common for room understanding)
# Run this cell to get the 10,000+ interior design images
!kaggle datasets download -d stepanyarullin/interior-design-styles
!unzip -q interior-design-styles.zip -d ./data



Dataset URL: https://www.kaggle.com/datasets/stepanyarullin/interior-design-styles
License(s): MIT
100%|█████████████████████████████████████████| 699M/699M [00:04<00:00, 161MB/s]



In [6]:
# Step 1: Create proper folders
!mkdir -p /kaggle/working/data/train
!mkdir -p /kaggle/working/data/val

# Step 2: Move training images
!mv /kaggle/working/data/dataset_train/dataset_train/* /kaggle/working/data/train/

# Step 3: Move validation images
!mv /kaggle/working/data/dataset_test/dataset_test/* /kaggle/working/data/val/

# Step 4: Clean up useless folders
!rm -rf /kaggle/working/data/dataset_train
!rm -rf /kaggle/working/data/dataset_test
!rm /kaggle/working/data/test_labels.csv

In [7]:
import os

print("Train:", os.listdir('/kaggle/working/data/train')[:5])
print("Val:", os.listdir('/kaggle/working/data/val')[:5])

Train: ['victorian', 'modern', 'mediterranean', 'traditional', 'french-country']
Val: ['victorian', 'modern', 'mediterranean', 'traditional', 'french-country']


In [9]:
from ultralytics import YOLO

model = YOLO('yolov8s-cls.pt')

model.train(
    data='/kaggle/working/data', 
    epochs=50,
    imgsz=320,
    batch=32
)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x785daf4e5280>
curves: []
curves_results: []
fitness: 0.5682488530874252
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.36953607201576233, 'metrics/accuracy_top5': 0.7669616341590881, 'fitness': 0.5682488530874252}
save_dir: PosixPath('/kaggle/working/runs/classify/train2')
speed: {'preprocess': 0.12262496540636214, 'inference': 0.5844348905878398, 'loss': 0.00022441056560683516, 'postprocess': 0.0004457516768373534}
top1: 0.36953607201576233
top5: 0.7669616341590881

In [11]:
import os

print(os.listdir('/kaggle/working/data/val/modern')[:10])

['modern_1004.jpg', 'modern_496.jpg', 'modern_454.jpg', 'modern_703.jpg', 'modern_864.jpg', 'modern_106.jpg', 'modern_325.jpg', 'modern_249.jpg', 'modern_375.jpg', 'modern_415.jpg']


In [12]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/classify/train2/weights/best.pt')

results = model('/kaggle/working/data/val/modern/modern_101.jpg')


image 1/1 /kaggle/working/data/val/modern/modern_101.jpg: 320x320 mid-century-modern 0.31, contemporary 0.29, asian 0.09, transitional 0.06, eclectic 0.05, 4.5ms
Speed: 5.5ms preprocess, 4.5ms inference, 0.1ms postprocess per image at shape (1, 3, 320, 320)


In [14]:
from ultralytics import YOLO
import os, random

# Load model
model = YOLO('/kaggle/working/runs/classify/train2/weights/best.pt')

# Pick a random image
folder = '/kaggle/working/data/val/modern'
img = random.choice(os.listdir(folder))

print("Testing:", img)

# Run prediction
results = model(f"{folder}/{img}")

# Extract probabilities
probs = results[0].probs

# Get class names
names = results[0].names

# Get top 5 predictions
top5 = probs.top5

print("\nTop Predictions:")
for i in top5:
    print(names[i])

Testing: modern_511.jpg

image 1/1 /kaggle/working/data/val/modern/modern_511.jpg: 320x320 asian 0.20, southwestern 0.12, mid-century-modern 0.11, contemporary 0.07, mediterranean 0.06, 3.6ms
Speed: 4.2ms preprocess, 3.6ms inference, 0.1ms postprocess per image at shape (1, 3, 320, 320)

Top Predictions:
asian
southwestern
mid-century-modern
contemporary
mediterranean


In [15]:
top3 = probs.top5[:3]

styles = [names[i] for i in top3]

print("Top 3 styles:", styles)

Top 3 styles: ['asian', 'southwestern', 'mid-century-modern']
